# Lekcija 08 - Vzorec načrtovanja večagentnega sistema


## Namestitev


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Zakaj večagentni sistemi?

Naloge iz resničnega sveta, kot je načrtovanje potovanja, vključujejo različna strokovna področja — logistiko, lokalno znanje, proračun in še več. Posamezen agent, ki poskuša upravljati vse, hitro postane nepraktičen.

Večagentni sistemi to rešujejo s **specializacijo**: vsak agent se osredotoči na eno področje strokovnega znanja in tako proizvede kakovostnejše rezultate kot generalist. Prav tako izboljšajo **razširljivost** — lahko dodate nove agente (npr. strokovnjaka za lete, restavratorskega kritika) brez ponovnega pisanja obstoječega poteka dela. Agenti delujejo skupaj preko strukturirane verige, pri čemer kontekst prenašajo iz enega na drugega.


## Ustvarjanje specializiranih agentov


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Gradnja zaporednega delovnega toka

`WorkflowBuilder` vam omogoča povezovanje agentov v usmerjen graf. Tukaj ustvarimo preprosto dvostopenjsko cevovod: **TravelPlanner** pripravi načrt poti, nato pa ga **TravelConcierge** pregleda in izboljša.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodajanje več agentov v delovni tok

Ena največjih prednosti vzorca večagentnega sistema je, kako enostavno ga je razširiti. Spodaj dodamo agenta **BudgetReviewer**, ki preveri načrt glede na proračun potnika, označi predmete, ki bi lahko povzročili preseganje stroškov, in predlaga denar prihranjevalne alternative. Delovni tok zdaj zažene tri agente zaporedoma:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Povzetek

V tej lekciji ste se naučili, kako:

1. **Ustvariti specializirane agente** — vsak s svojo osredotočeno vlogo (načrtovanje, posredovanje, pregled proračuna).
2. **Povezati agente v zaporedni potek dela** z uporabo `WorkflowBuilder` in `add_edge`.
3. **Pretakati izhod** iz večagentne verige, ob spremljanju, kateri agent govori.
4. **Razširiti potek dela** z dodajanjem novih agentov v verigo brez spreminjanja obstoječih.

Večagentni oblikovni vzorec ohranja vsak agent enostaven, hkrati pa ustvarja bogatejše in bolj temeljito pregledane rezultate, kot bi jih lahko dosegel posamezen agent sam.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
